# Chapitre 6 — Structuration et transformation

**Durée estimée : 8-10 heures**

---

## Objectifs d'apprentissage

À la fin de ce chapitre, vous serez capable de :

1. **Restructurer** des DataFrames avec pivot et melt selon les besoins d'analyse
2. **Combiner** des sources de données avec merge et concat en choisissant la bonne méthode
3. **Agréger** des données avec groupby et créer des statistiques par groupe
4. **Créer** de nouvelles variables pertinentes (feature engineering) pour l'analyse et le ML

---

## 🎯 Le Hook : Comment Spotify crée 10 000 features pour vous connaître

Spotify ne stocke pas simplement "vous avez écouté cette chanson". À partir de vos données d'écoute brutes, ils créent plus de **10 000 features** par utilisateur :

- Nombre d'écoutes par genre, par heure, par jour
- Tempo moyen préféré
- Diversité musicale (ratio nouveaux artistes / artistes habituels)
- Patterns saisonniers (musique de Noël en décembre ?)

Ces features transformées alimentent ensuite l'algorithme de recommandation.

La donnée brute ("user_123 a écouté track_456 à 14h32") devient une **représentation riche** de vos goûts.

C'est le pouvoir de la **transformation des données**.

> 💭 **Question Socratique #1** : Si Spotify stocke 10 000 features par utilisateur pour 500 millions d'utilisateurs, cela représente des milliards de données. Est-ce vraiment nécessaire de tout stocker, ou pourrait-on recalculer à la demande ?

---

# 📖 PARTIE THÉORIQUE

---

## 6.1 Restructuration des DataFrames

### Wide vs Long : deux visions des mêmes données

```
FORMAT WIDE (large)                    FORMAT LONG (long)
┌────────┬─────┬─────┬─────┐          ┌────────┬───────┬────────┐
│ Client │ Jan │ Fev │ Mar │          │ Client │ Mois  │ Ventes │
├────────┼─────┼─────┼─────┤          ├────────┼───────┼────────┤
│ Alice  │ 100 │ 150 │ 120 │    ↔     │ Alice  │ Jan   │ 100    │
│ Bob    │ 200 │ 180 │ 220 │          │ Alice  │ Fev   │ 150    │
└────────┴─────┴─────┴─────┘          │ Alice  │ Mar   │ 120    │
                                      │ Bob    │ Jan   │ 200    │
1 ligne par client                    │ Bob    │ Fev   │ 180    │
Colonnes = périodes                   │ Bob    │ Mar   │ 220    │
                                      └────────┴───────┴────────┘
                                      1 ligne par observation
```

### Quand utiliser quel format ?

| Format | Avantages | Cas d'usage |
|--------|-----------|-------------|
| **Wide** | Lisible, compact | Tableaux de reporting, Excel |
| **Long** | Flexible, analyses faciles | Visualisation, ML, base de données |

### Types de jointures (merge)

```
       INNER                LEFT                RIGHT               OUTER
    ┌─────────┐          ┌─────────┐          ┌─────────┐          ┌─────────┐
    │    A    │          │    A    │          │         │          │    A    │
    │  ┌───┐  │          │  ┌───┐  │          │  ┌───┐  │          │  ┌───┐  │
    │  │ X │  │          │  │ X │  │          │  │ X │  │          │  │ X │  │
    │  └───┘  │          │  └───┘  │          │  └───┘  │          │  └───┘  │
    │    B    │          │         │          │    B    │          │    B    │
    └─────────┘          └─────────┘          └─────────┘          └─────────┘

   Seulement ce        Tout A +            Tout B +             Tout A + Tout B
   qui matche          matchs de B         matchs de A
```

### Merge vs Concat

| Critère | Merge | Concat |
|---------|-------|--------|
| **Utilisation** | Joindre sur une clé | Empiler sans condition |
| **Colonnes** | Peuvent être différentes | Doivent correspondre (axis=0) |
| **Analogie SQL** | JOIN | UNION |
| **Cas typique** | Clients + Commandes | Janvier + Février + Mars |

### Le paradigme Split-Apply-Combine (GroupBy)

```
        DONNÉES ORIGINALES
        ┌──────────────────┐
        │ Region  Ventes   │
        │ Nord    100      │
        │ Sud     150      │
        │ Nord    120      │
        │ Sud     180      │
        │ Nord    90       │
        └──────────────────┘
              │ SPLIT
              ▼
    ┌─────────────┬─────────────┐
    │   Nord      │    Sud      │
    │   100       │    150      │
    │   120       │    180      │
    │   90        │             │
    └─────────────┴─────────────┘
              │ APPLY (sum)
              ▼
    ┌─────────────┬─────────────┐
    │   310       │    330      │
    └─────────────┴─────────────┘
              │ COMBINE
              ▼
        ┌──────────────────┐
        │ Region  Total    │
        │ Nord    310      │
        │ Sud     330      │
        └──────────────────┘
```

---

# 🖥️ PARTIE PRATIQUE

---

## 6.1 Restructuration des DataFrames (Pratique)

### Pivot : Long → Wide

In [ ]:
import pandas as pd
import numpy as np

# Données en format long
df_long = pd.DataFrame({
    'client': ['Alice', 'Alice', 'Alice', 'Bob', 'Bob', 'Bob'],
    'mois': ['Jan', 'Fev', 'Mar', 'Jan', 'Fev', 'Mar'],
    'ventes': [100, 150, 120, 200, 180, 220]
})

print("Format LONG :")
print(df_long)

In [ ]:
# Pivot vers format wide
df_wide = df_long.pivot(
    index='client',     # Devient les lignes
    columns='mois',     # Devient les colonnes
    values='ventes'     # Les valeurs à répartir
)

print("Format WIDE après pivot :")
print(df_wide)

### Pivot Table : avec agrégation

In [ ]:
# Quand il y a plusieurs valeurs par combinaison
df = pd.DataFrame({
    'client': ['Alice', 'Alice', 'Alice', 'Bob', 'Bob'],
    'categorie': ['A', 'A', 'B', 'A', 'B'],
    'montant': [100, 150, 200, 300, 250]
})

print("Données avec plusieurs valeurs par combinaison :")
print(df)

In [ ]:
# Pivot avec somme des montants
df_pivot = pd.pivot_table(
    df,
    index='client',
    columns='categorie',
    values='montant',
    aggfunc='sum',           # Fonction d'agrégation
    fill_value=0             # Remplacer NaN par 0
)

print("\nPivot table avec agrégation :")
print(df_pivot)

### ✍️ Exercice 6.1 : Pivot de ventes (10 min)

In [ ]:
import pandas as pd

# Ventes par région et trimestre
df = pd.DataFrame({
    'region': ['Nord', 'Nord', 'Nord', 'Nord', 'Sud', 'Sud', 'Sud', 'Sud'],
    'trimestre': ['Q1', 'Q2', 'Q3', 'Q4', 'Q1', 'Q2', 'Q3', 'Q4'],
    'ventes': [1000, 1200, 1100, 1500, 800, 900, 850, 1000]
})

print("Données originales :")
print(df)

In [ ]:
# 1. Créez un tableau pivot : régions en lignes, trimestres en colonnes
df_pivot = df.pivot(
    index='region',
    columns='trimestre',
    values='ventes'
)
print("Pivot simple :")
print(df_pivot)

In [ ]:
# 2. Ajoutez une colonne 'Total' avec la somme des trimestres
df_pivot['Total'] = df_pivot.sum(axis=1)
print("\nAvec total :")
print(df_pivot)

# 3. Quelle région a le meilleur Q4 ?
print(f"\nMeilleur Q4 : {df_pivot['Q4'].idxmax()} avec {df_pivot['Q4'].max()}")

### Melt : Wide → Long

In [ ]:
# Données en format wide
df_wide = pd.DataFrame({
    'client': ['Alice', 'Bob'],
    'Jan': [100, 200],
    'Fev': [150, 180],
    'Mar': [120, 220]
})

print("Format WIDE :")
print(df_wide)

In [ ]:
# Melt vers format long
df_long = pd.melt(
    df_wide,
    id_vars=['client'],           # Colonnes à garder fixes
    value_vars=['Jan', 'Fev', 'Mar'],  # Colonnes à "fondre"
    var_name='mois',              # Nom de la nouvelle colonne de variables
    value_name='ventes'           # Nom de la nouvelle colonne de valeurs
)

print("Format LONG après melt :")
print(df_long)

### ✍️ Exercice 6.2 : Melt pour visualisation (10 min)

In [ ]:
import pandas as pd

# Données de performance au format wide
df_wide = pd.DataFrame({
    'employe': ['Alice', 'Bob', 'Charlie'],
    'score_2022': [85, 78, 92],
    'score_2023': [88, 82, 90],
    'score_2024': [91, 85, 95]
})

print("Format wide :")
print(df_wide)

In [ ]:
# Transformez en format long pour pouvoir créer un graphique de l'évolution
df_long = pd.melt(
    df_wide,
    id_vars=['employe'],
    value_vars=['score_2022', 'score_2023', 'score_2024'],
    var_name='annee',
    value_name='score'
)

# Nettoyez la colonne année pour extraire uniquement l'année
df_long['annee'] = df_long['annee'].str.replace('score_', '').astype(int)

print("\nFormat long (prêt pour visualisation) :")
print(df_long)

> 💭 **Question Socratique #2** : Pourquoi les bibliothèques de visualisation comme Seaborn préfèrent-elles généralement le format long ? Quel avantage cela apporte-t-il pour créer des graphiques ?

---

## 6.2 Combinaison de données

### Merge : joindre sur une clé

In [ ]:
# Création des tables
clients = pd.DataFrame({
    'client_id': [1, 2, 3],
    'nom': ['Alice', 'Bob', 'Charlie'],
    'ville': ['Paris', 'Lyon', 'Marseille']
})

commandes = pd.DataFrame({
    'commande_id': [101, 102, 103, 104],
    'client_id': [1, 1, 2, 4],  # Note: client 4 n'existe pas dans clients
    'montant': [100, 150, 200, 50]
})

print("Table clients :")
print(clients)
print("\nTable commandes :")
print(commandes)

In [ ]:
# Inner join (intersection) - seulement ce qui matche
df_inner = pd.merge(clients, commandes, on='client_id', how='inner')
print("INNER JOIN :")
print(df_inner)

In [ ]:
# Left join (tous les clients)
df_left = pd.merge(clients, commandes, on='client_id', how='left')
print("LEFT JOIN (tous les clients) :")
print(df_left)

In [ ]:
# Right join (toutes les commandes)
df_right = pd.merge(clients, commandes, on='client_id', how='right')
print("RIGHT JOIN (toutes les commandes) :")
print(df_right)

In [ ]:
# Outer join (union)
df_outer = pd.merge(clients, commandes, on='client_id', how='outer')
print("OUTER JOIN (union) :")
print(df_outer)

### Cas courants de merge

In [ ]:
# Tables avec noms de colonnes différents
clients_v2 = pd.DataFrame({
    'id_client': [1, 2, 3],
    'nom': ['Alice', 'Bob', 'Charlie']
})

commandes_v2 = pd.DataFrame({
    'customer_id': [1, 1, 2],
    'montant': [100, 150, 200]
})

# Merge avec clés différentes
df_merge = pd.merge(
    clients_v2,
    commandes_v2,
    left_on='id_client',
    right_on='customer_id',
    how='left'
)
print("Merge avec clés différentes :")
print(df_merge)

In [ ]:
# Merge sur plusieurs colonnes
ventes = pd.DataFrame({
    'annee': [2023, 2023, 2024, 2024],
    'region': ['Nord', 'Sud', 'Nord', 'Sud'],
    'ventes': [1000, 1200, 1100, 1300]
})

objectifs = pd.DataFrame({
    'annee': [2023, 2023, 2024, 2024],
    'region': ['Nord', 'Sud', 'Nord', 'Sud'],
    'objectif': [950, 1100, 1050, 1250]
})

df_merge_multi = pd.merge(ventes, objectifs, on=['annee', 'region'], how='left')
df_merge_multi['ecart'] = df_merge_multi['ventes'] - df_merge_multi['objectif']

print("Merge sur plusieurs colonnes :")
print(df_merge_multi)

### ✍️ Exercice 6.3 : Jointures multiples (15 min)

In [ ]:
import pandas as pd

# Trois tables à joindre
clients = pd.DataFrame({
    'client_id': [1, 2, 3, 4],
    'nom': ['Alice', 'Bob', 'Charlie', 'David'],
    'segment': ['Premium', 'Standard', 'Premium', 'Standard']
})

commandes = pd.DataFrame({
    'commande_id': [101, 102, 103, 104, 105],
    'client_id': [1, 1, 2, 5, 3],  # client 5 n'existe pas
    'produit_id': [10, 20, 10, 30, 20],
    'quantite': [2, 1, 5, 3, 2]
})

produits = pd.DataFrame({
    'produit_id': [10, 20, 30],
    'nom_produit': ['Widget A', 'Widget B', 'Widget C'],
    'prix': [50, 75, 100]
})

print("Table clients :")
print(clients)
print("\nTable commandes :")
print(commandes)
print("\nTable produits :")
print(produits)

In [ ]:
# Étape 1 : Joindre commandes et clients (left join sur commandes)
df = pd.merge(commandes, clients, on='client_id', how='left')
print("Après jointure commandes + clients :")
print(df)

In [ ]:
# Étape 2 : Joindre avec produits
df = pd.merge(df, produits, on='produit_id', how='left')
print("\nAprès jointure avec produits :")
print(df)

In [ ]:
# Étape 3 : Calculer le montant total par commande
df['montant'] = df['quantite'] * df['prix']

# Étape 4 : Afficher les commandes du segment Premium
print("\nCommandes Premium :")
print(df[df['segment'] == 'Premium'])

# Question : La commande 104 (client_id=5) apparaît-elle dans le résultat ?
print("\n→ La commande 104 apparaît mais avec NaN pour le client (client_id=5 n'existe pas)")

### Concat : empiler des DataFrames

In [ ]:
# Empiler verticalement (ajout de lignes)
df_jan = pd.DataFrame({'mois': ['Jan']*3, 'ventes': [100, 120, 90]})
df_fev = pd.DataFrame({'mois': ['Fev']*3, 'ventes': [110, 130, 95]})
df_mar = pd.DataFrame({'mois': ['Mar']*3, 'ventes': [105, 125, 100]})

df_q1 = pd.concat([df_jan, df_fev, df_mar], ignore_index=True)
print("Concat vertical (ignore_index=True) :")
print(df_q1)

In [ ]:
# Avec identification de l'origine
df_all = pd.concat(
    [df_jan, df_fev, df_mar],
    keys=['Janvier', 'Février', 'Mars']
)
print("\nConcat avec keys (identifiant d'origine) :")
print(df_all)

### ✍️ Exercice 6.4 : Concat de fichiers mensuels (10 min)

In [ ]:
import pandas as pd
import numpy as np

# Simuler des fichiers mensuels
def creer_donnees_mois(mois, n=30):
    np.random.seed(mois)
    return pd.DataFrame({
        'date': pd.date_range(f'2024-{mois:02d}-01', periods=n, freq='D')[:n],
        'ventes': np.random.randint(100, 500, n),
        'region': np.random.choice(['Nord', 'Sud', 'Est', 'Ouest'], n)
    })

df_jan = creer_donnees_mois(1)
df_fev = creer_donnees_mois(2)
df_mar = creer_donnees_mois(3)

print(f"Janvier : {len(df_jan)} lignes")
print(f"Février : {len(df_fev)} lignes")
print(f"Mars : {len(df_mar)} lignes")

In [ ]:
# 1. Concaténer les trois mois
df_q1 = pd.concat([df_jan, df_fev, df_mar], ignore_index=True)
print(f"\nQ1 total : {len(df_q1)} lignes")

In [ ]:
# 2. Ajouter une colonne pour identifier le mois d'origine
df_jan['mois'] = 'Janvier'
df_fev['mois'] = 'Février'
df_mar['mois'] = 'Mars'
df_q1_tagged = pd.concat([df_jan, df_fev, df_mar], ignore_index=True)

# 3. Calculer les ventes totales par mois
print("\nVentes par mois :")
print(df_q1_tagged.groupby('mois')['ventes'].sum())

> 💭 **Question Socratique #3** : Vous devez combiner des données clients provenant de deux systèmes différents (CRM et e-commerce). Les deux ont une colonne "client_id" mais les ID ne sont pas les mêmes. Comment aborderiez-vous ce problème ?

---

## 6.3 Agrégations avec GroupBy

In [ ]:
# Données de ventes
np.random.seed(42)
df = pd.DataFrame({
    'region': np.random.choice(['Nord', 'Sud', 'Est', 'Ouest'], 100),
    'categorie': np.random.choice(['A', 'B', 'C'], 100),
    'ventes': np.random.randint(50, 500, 100),
    'client_id': np.random.randint(1, 30, 100)
})

print("Données :")
print(df.head(10))

In [ ]:
# Agrégation simple
print("Ventes totales par région :")
print(df.groupby('region')['ventes'].sum())

In [ ]:
# Plusieurs colonnes de groupement
print("\nVentes par région et catégorie :")
print(df.groupby(['region', 'categorie'])['ventes'].sum().unstack())

In [ ]:
# Plusieurs fonctions d'agrégation
print("\nStatistiques par région :")
print(df.groupby('region')['ventes'].agg(['sum', 'mean', 'count', 'std']).round(2))

In [ ]:
# Agrégations nommées (syntaxe moderne)
df_agg = df.groupby('region').agg(
    total_ventes=('ventes', 'sum'),
    moyenne_ventes=('ventes', 'mean'),
    nb_transactions=('ventes', 'count'),
    nb_clients=('client_id', 'nunique')
).round(2).reset_index()

print("\nAgrégations nommées :")
print(df_agg)

### Fonctions d'agrégation courantes

| Fonction | Description |
|----------|-------------|
| `sum()` | Somme |
| `mean()` | Moyenne |
| `median()` | Médiane |
| `count()` | Nombre de valeurs non-null |
| `size()` | Nombre total de lignes |
| `nunique()` | Nombre de valeurs uniques |
| `min()`, `max()` | Minimum, Maximum |
| `std()`, `var()` | Écart-type, Variance |
| `first()`, `last()` | Première, Dernière valeur |

### ✍️ Exercice 6.5 : Analyse par groupe (15 min)

In [ ]:
import pandas as pd
import numpy as np

# Données de ventes
np.random.seed(42)
df = pd.DataFrame({
    'date': pd.date_range('2024-01-01', periods=1000, freq='D'),
    'region': np.random.choice(['Nord', 'Sud', 'Est', 'Ouest'], 1000),
    'categorie': np.random.choice(['Électronique', 'Vêtements', 'Alimentation'], 1000),
    'montant': np.random.randint(10, 500, 1000),
    'client_id': np.random.randint(1, 200, 1000)
})
df['mois'] = df['date'].dt.month

print(f"Dataset : {len(df)} lignes")

In [ ]:
# 1. Ventes totales par région
print("Ventes par région :")
print(df.groupby('region')['montant'].sum())

In [ ]:
# 2. Ventes moyennes par catégorie et région
print("\nVentes moyennes par catégorie et région :")
print(df.groupby(['categorie', 'region'])['montant'].mean().round(2).unstack())

In [ ]:
# 3. Nombre de clients uniques par région
print("\nClients uniques par région :")
print(df.groupby('region')['client_id'].nunique())

In [ ]:
# 4. Top 3 des mois par ventes totales
print("\nTop 3 mois :")
ventes_mois = df.groupby('mois')['montant'].sum().sort_values(ascending=False)
print(ventes_mois.head(3))

In [ ]:
# 5. Rapport complet par région
rapport = df.groupby('region').agg(
    total_ventes=('montant', 'sum'),
    moyenne_ventes=('montant', 'mean'),
    nb_transactions=('montant', 'count'),
    nb_clients=('client_id', 'nunique')
).round(2)

print("\nRapport complet :")
print(rapport)

---

## 6.4 Feature Engineering basique

### Qu'est-ce que le Feature Engineering ?

Le **feature engineering** consiste à créer de nouvelles variables (features) à partir des données existantes pour améliorer l'analyse ou les performances des modèles ML.

In [ ]:
# Variables dérivées (calculs simples)
df_fe = pd.DataFrame({
    'produit_id': [1, 2, 3],
    'prix_vente': [100, 150, 200],
    'cout': [60, 90, 120],
    'quantite_vendue': [50, 30, 20]
})

# Calculs
df_fe['marge'] = df_fe['prix_vente'] - df_fe['cout']
df_fe['taux_marge'] = (df_fe['marge'] / df_fe['prix_vente'] * 100).round(2)
df_fe['ca_total'] = df_fe['prix_vente'] * df_fe['quantite_vendue']

print("Variables dérivées :")
print(df_fe)

### Variables temporelles

In [ ]:
# À partir d'une date
df_temps = pd.DataFrame({
    'commande_id': [1, 2, 3, 4, 5],
    'date': pd.to_datetime(['2024-01-15 10:30', '2024-02-20 14:45', 
                           '2024-03-25 18:20', '2024-04-06 09:00', '2024-05-12 22:15'])
})

# Extraction de composantes
df_temps['annee'] = df_temps['date'].dt.year
df_temps['mois'] = df_temps['date'].dt.month
df_temps['jour'] = df_temps['date'].dt.day
df_temps['jour_semaine'] = df_temps['date'].dt.dayofweek  # 0=lundi
df_temps['heure'] = df_temps['date'].dt.hour
df_temps['trimestre'] = df_temps['date'].dt.quarter

print("Composantes temporelles :")
print(df_temps)

In [ ]:
# Variables binaires temporelles
df_temps['est_weekend'] = df_temps['jour_semaine'].isin([5, 6]).astype(int)
df_temps['est_soiree'] = (df_temps['heure'] >= 18).astype(int)

print("\nAvec variables binaires :")
print(df_temps[['date', 'jour_semaine', 'est_weekend', 'heure', 'est_soiree']])

### ✍️ Exercice 6.6 : Feature engineering temporel (15 min)

In [ ]:
import pandas as pd
import numpy as np

# Données de commandes
np.random.seed(42)
df = pd.DataFrame({
    'commande_id': range(1, 101),
    'date_commande': pd.date_range('2024-01-01', periods=100, freq='4H'),
    'client_id': np.random.randint(1, 20, 100),
    'montant': np.random.randint(20, 500, 100)
})

print("Données initiales :")
print(df.head())

In [ ]:
# 1. Extraire les composantes temporelles
df['annee'] = df['date_commande'].dt.year
df['mois'] = df['date_commande'].dt.month
df['jour_semaine'] = df['date_commande'].dt.dayofweek
df['heure'] = df['date_commande'].dt.hour

# 2. Créer des features binaires
df['est_weekend'] = df['jour_semaine'].isin([5, 6]).astype(int)
df['est_soiree'] = (df['heure'] >= 18).astype(int)
df['est_nuit'] = ((df['heure'] >= 22) | (df['heure'] <= 6)).astype(int)

print("\nAvec features temporelles :")
print(df[['date_commande', 'jour_semaine', 'est_weekend', 'heure', 'est_soiree']].head(10))

In [ ]:
# 3. Analyser les patterns
print("Commandes par jour de semaine :")
print(df.groupby('jour_semaine')['commande_id'].count())

print("\nMontant moyen : weekend vs semaine")
print(df.groupby('est_weekend')['montant'].mean().round(2))

In [ ]:
# Commandes par tranche horaire
df['tranche'] = pd.cut(df['heure'], bins=[0, 6, 12, 18, 24], 
                       labels=['Nuit', 'Matin', 'Après-midi', 'Soir'])
print("\nCommandes par tranche horaire :")
print(df.groupby('tranche', observed=True)['commande_id'].count())

### Binning / Discrétisation

In [ ]:
# Binning avec pd.cut (intervalles égaux)
df_bin = pd.DataFrame({
    'client_id': range(1, 11),
    'age': [22, 35, 45, 28, 67, 52, 38, 19, 73, 41],
    'salaire': [25000, 45000, 55000, 32000, 85000, 62000, 48000, 18000, 95000, 51000]
})

# Binning par intervalles définis
df_bin['tranche_age'] = pd.cut(
    df_bin['age'],
    bins=[0, 25, 35, 50, 65, 100],
    labels=['18-25', '26-35', '36-50', '51-65', '65+']
)

print("Binning par intervalles :")
print(df_bin[['client_id', 'age', 'tranche_age']])

In [ ]:
# Binning avec pd.qcut (quantiles)
df_bin['quartile_salaire'] = pd.qcut(
    df_bin['salaire'],
    q=4,
    labels=['Q1', 'Q2', 'Q3', 'Q4']
)

print("\nBinning par quantiles :")
print(df_bin[['client_id', 'salaire', 'quartile_salaire']])

### Encoding catégoriel

In [ ]:
# One-Hot Encoding
df_cat = pd.DataFrame({
    'client_id': [1, 2, 3, 4],
    'region': ['Nord', 'Sud', 'Est', 'Ouest'],
    'segment': ['Premium', 'Standard', 'Premium', 'Basic']
})

df_encoded = pd.get_dummies(df_cat, columns=['region'], prefix='reg')
print("One-Hot Encoding :")
print(df_encoded)

In [ ]:
# Label Encoding (convertir en nombres)
df_cat['region_code'] = df_cat['region'].astype('category').cat.codes
print("\nLabel Encoding :")
print(df_cat[['region', 'region_code']])

In [ ]:
# Mapping manuel
mapping_segment = {'Basic': 1, 'Standard': 2, 'Premium': 3}
df_cat['segment_score'] = df_cat['segment'].map(mapping_segment)
print("\nMapping manuel :")
print(df_cat[['segment', 'segment_score']])

### ✍️ Exercice 6.7 : Pipeline de feature engineering (20 min)

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

# Données clients
np.random.seed(42)
df = pd.DataFrame({
    'client_id': range(1, 101),
    'date_naissance': pd.date_range('1960-01-01', periods=100, freq='150D'),
    'date_inscription': pd.date_range('2020-01-01', periods=100, freq='10D'),
    'region': np.random.choice(['Nord', 'Sud', 'Est', 'Ouest'], 100),
    'nb_commandes': np.random.randint(0, 50, 100),
    'montant_total': np.random.randint(0, 5000, 100)
})

print("Données initiales :")
print(df.head())
print(f"\nNombre de colonnes : {len(df.columns)}")

In [ ]:
# 1. Calculer l'âge
df['age'] = (datetime.now() - df['date_naissance']).dt.days // 365

# 2. Créer des tranches d'âge
df['tranche_age'] = pd.cut(df['age'], bins=[0, 30, 45, 60, 100], 
                           labels=['<30', '30-45', '45-60', '60+'])

# 3. Calculer l'ancienneté en mois
df['anciennete_mois'] = (datetime.now() - df['date_inscription']).dt.days // 30

# 4. Calculer le panier moyen (attention à la division par 0)
df['panier_moyen'] = np.where(df['nb_commandes'] > 0, 
                              df['montant_total'] / df['nb_commandes'], 0)

# 5. Créer un flag "client actif" (au moins 5 commandes)
df['est_actif'] = (df['nb_commandes'] >= 5).astype(int)

# 6. Créer des quartiles de montant total
df['segment_valeur'] = pd.qcut(df['montant_total'], q=4, 
                               labels=['Bronze', 'Silver', 'Gold', 'Platinum'])

print("\nAprès feature engineering :")
print(df[['client_id', 'age', 'tranche_age', 'anciennete_mois', 
          'panier_moyen', 'est_actif', 'segment_valeur']].head(10))

In [ ]:
# 7. One-hot encoding de la région
df_final = pd.get_dummies(df, columns=['region'], prefix='reg')

print("\nDataset final :")
print(f"Nombre de colonnes : {len(df_final.columns)}")
print(f"Colonnes : {list(df_final.columns)}")

---

## 🧠 Réflexion métacognitive

### Auto-évaluation

| Compétence | 1 | 2 | 3 | 4 | 5 |
|------------|---|---|---|---|---|
| Je sais utiliser pivot et melt | ○ | ○ | ○ | ○ | ○ |
| Je maîtrise les différents types de merge | ○ | ○ | ○ | ○ | ○ |
| Je peux créer des agrégations avec groupby | ○ | ○ | ○ | ○ | ○ |
| Je sais créer des features temporelles | ○ | ○ | ○ | ○ | ○ |
| Je peux encoder des variables catégorielles | ○ | ○ | ○ | ○ | ○ |

### Questions de réflexion

1. **Quel type de merge** utilisez-vous le plus souvent dans votre pratique ?

2. **Quelles features** créeriez-vous à partir d'une date de naissance ?

3. **Comment décideriez-vous** entre one-hot encoding et label encoding ?

---

## 📚 Résumé du chapitre

### Points clés à retenir

1. **Restructuration** :
   - `pivot` : Long → Wide (pour reporting)
   - `melt` : Wide → Long (pour visualisation)

2. **Combinaison** :
   - `merge` : Jointure sur clé (inner, left, right, outer)
   - `concat` : Empilement (vertical ou horizontal)

3. **Agrégation** :
   - `groupby` + `agg` pour statistiques par groupe
   - Fonctions : sum, mean, count, nunique...

4. **Feature Engineering** :
   - Variables calculées (ratios, différences)
   - Variables temporelles (année, mois, jour_semaine)
   - Binning (cut, qcut)
   - Encoding (get_dummies, cat.codes)

---

## ➡️ Prochain chapitre

**Chapitre 7 : EDA analytique** — Vous apprendrez à comprendre vos données propres, identifier des patterns et formuler des hypothèses.

---

*Module 2 — Pipeline Data | Chapitre 6 sur 11*